Рассмотрим 2 подхода:

1) Генерация отдельных фраз

Данные в начале очищаем от лишних символов
   
target = текст отстоящий от текущего на sequence_length

Модель - lstm

Оптимайзер - константный Adam, lr=0.001

Добавлена подрезка нормы градиента

loss - CrossEntropyLoss

2) Генерация по буквам

Данные разбиваем по отдельным символам(оставляем табы для сохранения стилистики)

target = текст отстоящий от текущего на sequence_length, sequence_length теперь больше, так как до этого были фразы, а теперь буквы

Модель - lstm

Оптимайзер - Adam с изменением по ReduceLROnPlateau

loss - CrossEntropyLoss

# Генерация фраз

In [235]:
path_to_source = "C:\\Users\\aleksandr.egorov\\Downloads\\onegin.txt"

In [152]:
from string import digits, whitespace

cyrillic_letters = u"абвгдеёжзийклмнопрстуфхцчшщъыьэюяАБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ"


def strip(text):
    allowed_chars = cyrillic_letters + digits + whitespace
    return "".join([c for c in text if c in allowed_chars])

In [250]:

config = {"max_epochs":30,
       "batch_size":256,
       "sequence_length":5}
dataset = Dataset(config)

In [220]:
import torch
import pandas as pd
from collections import Counter

class CustomDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        config,
    ):
        self.config = config
        self.words = self.load_words()
        self.uniq_words = self.get_uniq_words()

        self.index_to_word = {index: word for index, word in enumerate(self.uniq_words)}
        self.word_to_index = {word: index for index, word in enumerate(self.uniq_words)}

        self.words_indexes = [self.word_to_index[w] for w in self.words]

    def load_words(self):
        with open(path_to_source, 'r', encoding = 'utf-8' ) as file:
            data = file.read().lower().replace("\n", "").replace("\t", "")
        return [strip(i) for i in data if i != ""][1:]

    def get_uniq_words(self):
        word_counts = Counter(self.words)
        return sorted(word_counts, key=word_counts.get, reverse=True)

    def __len__(self):
        return len(self.words_indexes) - self.config["sequence_length"]

    def __getitem__(self, index):
        return (
            torch.tensor(self.words_indexes[index:index+self.config["sequence_length"]]),
            torch.tensor(self.words_indexes[index+1:index+self.config["sequence_length"]+1]),
        )

In [223]:
import torch
from torch import nn

class PoetryModel(nn.Module):
    def __init__(self, dataset):
        super(Model, self).__init__()
        self.hidden_size = 128
        self.embedding_dim = 128
        self.num_layers = 3

        self.embedding = nn.Embedding(
            num_embeddings=len(dataset.uniq_words),
            embedding_dim=self.embedding_dim,
        )
        self.lstm = nn.LSTM(
            input_size=self.hidden_size,
            hidden_size=self.hidden_size,
            num_layers=self.num_layers,
            dropout=0.1,
        )
        self.fc = nn.Linear(self.hidden_size, n_vocab)

    def forward(self, x, prev_state):
        emb = self.embedding(x)
        output, state = self.lstm(emb, prev_state)
        logits = self.fc(output)

        return logits, state

    def init_state(self, sequence_length):
        return (torch.zeros(self.num_layers, sequence_length, self.hidden_size),
                torch.zeros(self.num_layers, sequence_length, self.hidden_size))

In [221]:
def train(dataset, model, config):
    model.train()

    dataloader = DataLoader(
        dataset,
        batch_size=config["batch_size"],
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(config["max_epochs"]):
        state_h, state_c = model.init_state(config["sequence_length"])

        for batch, (x, y) in enumerate(dataloader):

            optimizer.zero_grad()

            y_pred, (state_h, state_c) = model(x, (state_h, state_c))
            loss = criterion(y_pred.transpose(1, 2), y)

            state_h = state_h.detach()
            state_c = state_c.detach()

            loss.backward()
            optimizer.step()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
            print({ 'epoch': epoch, 'batch': batch, 'loss': loss.item() })

def predict(dataset, model, text, next_words=100):
    words = text.split('/t')
    model.eval()

    state_h, state_c = model.init_state(len(words))

    for i in range(0, next_words):
        print(dataset.get_uniq_words())
        x = torch.tensor([[dataset.word_to_index[w] for w in words[i:]]])
        y_pred, (state_h, state_c) = model(x, (state_h, state_c))

        last_word_logits = y_pred[0][-1]
        p = torch.nn.functional.softmax(last_word_logits, dim=0).detach().numpy()
        word_index = np.random.choice(len(last_word_logits), p=p)
        words.append(dataset.index_to_word[word_index])

    return words

In [222]:
import torch
import numpy as np
from torch import nn, optim
from torch.utils.data import DataLoader

dataset = CustomDataset(config)
model = PoetryModel(dataset)

train(dataset, model, config)


{'epoch': 0, 'batch': 0, 'loss': 8.571271896362305}
{'epoch': 0, 'batch': 1, 'loss': 8.567377090454102}
{'epoch': 0, 'batch': 2, 'loss': 8.559419631958008}
{'epoch': 0, 'batch': 3, 'loss': 8.566705703735352}
{'epoch': 0, 'batch': 4, 'loss': 8.570815086364746}
{'epoch': 0, 'batch': 5, 'loss': 8.567146301269531}
{'epoch': 0, 'batch': 6, 'loss': 8.573144912719727}
{'epoch': 0, 'batch': 7, 'loss': 8.58089542388916}
{'epoch': 0, 'batch': 8, 'loss': 8.577558517456055}
{'epoch': 0, 'batch': 9, 'loss': 8.577859878540039}
{'epoch': 0, 'batch': 10, 'loss': 8.584598541259766}
{'epoch': 0, 'batch': 11, 'loss': 8.578027725219727}
{'epoch': 0, 'batch': 12, 'loss': 8.586509704589844}
{'epoch': 0, 'batch': 13, 'loss': 8.579209327697754}
{'epoch': 0, 'batch': 14, 'loss': 8.573036193847656}
{'epoch': 0, 'batch': 15, 'loss': 8.579263687133789}
{'epoch': 0, 'batch': 16, 'loss': 8.581953048706055}
{'epoch': 0, 'batch': 17, 'loss': 8.582307815551758}
{'epoch': 0, 'batch': 18, 'loss': 8.577260971069336}
{'ep

In [204]:
predict(dataset, model, text='его пример другим наука')

['его пример другим наука',
 'пора вставать седьмой уж час',
 'с холма господский видит дом',
 'всё указует на нее',
 'так видно бог велел мой ваня',
 'в слезах раскашлялась она',
 'за ним гналася в шумном свете',
 'где льется светлый ручеек',
 'всё чувства поражает вдруг',
 'унижусь до смиренной прозы',
 'галоп мазурка вальс меж тем']

In [233]:
predict(dataset, model, text='мой дядя самых честных правил')

['мой дядя самых честных правил',
 'и умного дурачить славно',
 'течет так тихо так согласно',
 'для призраков закрыл я вежды',
 'и край отцов и заточенье',
 'или задумчивый вампир',
 'покамест моего романа',
 'татьяна бедная горит',
 'на ветви сосны преклоненной',
 'цензуре долг свой заплачу',
 'меж тем цель оды высока']

In [234]:
predict(dataset, model, text='татьяна бедная горит')

['татьяна бедная горит',
 'садится таня у окна',
 'читал охотно апулея',
 'но звон брегета им доносит',
 'дней несколько она потом',
 'залить горячий жир котлет',
 'лихая мода наш тиран',
 'он с лирой странствовал на свете',
 'неблагосклонно говорят',
 'почетный гражданин кулис',
 'на шум блистательных сует']

# Генерация символов

In [252]:
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [253]:
with open(path_to_source, 'r', encoding = 'utf-8' ) as file:
    data = file.readlines()
data = ' '.join(data)


def text_to_seq(data):
    char_counts = Counter(data)
    char_counts = sorted(char_counts.items(), key = lambda x: x[1], reverse=True)

    sorted_chars = [char for char, _ in char_counts]
    char_to_idx = {char: index for index, char in enumerate(sorted_chars)}
    idx_to_char = {v: k for k, v in char_to_idx.items()}
    sequence = np.array([char_to_idx[char] for char in data])
    
    return sequence, char_to_idx, idx_to_char

sequence, char_to_idx, idx_to_char = text_to_seq(data)

In [254]:
SEQ_LEN = 256

def get_batch(sequence):
    trains = []
    targets = []
    for _ in range(config["batch_size"]):
        batch_start = np.random.randint(0, len(sequence) - SEQ_LEN)
        chunk = sequence[batch_start: batch_start + SEQ_LEN]
        train = torch.LongTensor(chunk[:-1]).view(-1, 1)
        target = torch.LongTensor(chunk[1:]).view(-1, 1)
        trains.append(train)
        targets.append(target)
    return torch.stack(trains, dim=0), torch.stack(targets, dim=0)

In [255]:
def evaluate(model, char_to_idx, idx_to_char, start_text='.', prediction_len=200, temp=0.5):
    hidden = model.init_hidden()
    idx_input = [char_to_idx[char] for char in start_text]
    train = torch.LongTensor(idx_input).view(-1, 1, 1).to(device)
    predicted_text = start_text
    
    _, hidden = model(train, hidden)
        
    inp = train[-1].view(-1, 1, 1)
    for i in range(prediction_len):
        output, hidden = model(inp.to(device), hidden)
        output_logits = output.cpu().data.view(-1)
        p_next = F.softmax(output_logits / temp, dim=-1).detach().cpu().data.numpy()        
        top_index = np.random.choice(len(char_to_idx), p=p_next)
        inp = torch.LongTensor([top_index]).view(-1, 1, 1).to(device)
        predicted_char = idx_to_char[top_index]
        predicted_text += predicted_char
    
    return predicted_text

In [256]:
class CharPoetryModel(nn.Module):
    
    def __init__(self, input_size, hidden_size, embedding_size, n_layers=3):
        super().__init__()
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.embedding_size = embedding_size
        self.n_layers = n_layers

        self.encoder = nn.Embedding(self.input_size, self.embedding_size)
        self.lstm = nn.LSTM(self.embedding_size, self.hidden_size, self.n_layers)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(self.hidden_size, self.input_size)
        
    def forward(self, x, hidden):
        x = self.encoder(x).squeeze(2)
        out, (ht1, ct1) = self.lstm(x, hidden)
        out = self.dropout(out)
        x = self.fc(out)
        return x, (ht1, ct1)
    
    def init_hidden(self, batch_size=1):
        return (torch.zeros(self.n_layers, batch_size, self.hidden_size, requires_grad=True).to(device),
               torch.zeros(self.n_layers, batch_size, self.hidden_size, requires_grad=True).to(device))

In [257]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = CharPoetryModel(input_size=len(idx_to_char), hidden_size=128, embedding_size=128, n_layers=3)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, amsgrad=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    patience=5, 
    verbose=True, 
    factor=0.5
)

n_epochs = 10000
loss_avg = []

for epoch in range(n_epochs):
    model.train()
    train, target = get_batch(sequence)
    train = train.permute(1, 0, 2).to(device)
    target = target.permute(1, 0, 2).to(device)
    hidden = model.init_hidden(config["batch_size"])

    output, hidden = model(train, hidden)
    loss = criterion(output.permute(1, 2, 0), target.squeeze(-1).permute(1, 0))
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    loss_avg.append(loss.item())
    if len(loss_avg) >= 50:
        mean_loss = np.mean(loss_avg)
        print(f'Loss: {mean_loss}')
        scheduler.step(mean_loss)
        loss_avg = []
        model.eval()
        predicted_text = evaluate(model, char_to_idx, idx_to_char)

Loss: 3.4918439388275146
Loss: 3.382441987991333
Loss: 3.0865349054336546
Loss: 2.818454132080078
Loss: 2.489473509788513
Loss: 2.3316631746292114
Loss: 2.2303047370910645
Loss: 2.11965548992157
Loss: 2.042138352394104
Loss: 1.9713800263404846
Loss: 1.919326832294464
Loss: 1.8645414566993714
Loss: 1.8158324646949768
Loss: 1.7788118052482604
Loss: 1.7451992845535278
Loss: 1.6968843817710877
Loss: 1.6560768747329713
Loss: 1.6280368065834045
Loss: 1.5817493629455566
Loss: 1.5452725100517273
Loss: 1.5089247226715088
Loss: 1.4756213521957398
Loss: 1.439814908504486
Loss: 1.4121374249458314
Loss: 1.3838646030426025
Loss: 1.3571619963645936
Loss: 1.330109806060791
Loss: 1.3064041185379027
Loss: 1.282321617603302
Loss: 1.2585682010650634
Loss: 1.2394600248336791
Loss: 1.2173596239089965
Loss: 1.1978598165512084
Loss: 1.1737190198898315
Loss: 1.1614872241020202
Loss: 1.1452474641799926
Loss: 1.1220791220664978
Loss: 1.1101377940177917
Loss: 1.0968873119354248
Loss: 1.080301969051361
Loss: 1.060

In [258]:
model.eval()

print(evaluate(
    model, 
    char_to_idx, 
    idx_to_char, 
    temp=0.3, 
    prediction_len=500, 
    start_text='V'
    )
)

VI
 
 		Когда б он знал, какая раней
 		Уж по мне не сповени
 		Мои сердце в слог на призсково,
 		Волненье сердце мугой,
 		Печальных много прежни запижень.
 
 
 
 XLI
 
 		Встает девость забыть от кокужотом
 		И что ж пучка и емрасной
 		Облану соседуя расскажемя;
 		Стальки модности предрассыон
 		Воходил он проказник принить
 		И, по обычаю народа,
 		Сволитатель в ужанялся? Надь в садеженья
 		И наконец от них отпрадумани,
 		И я оскорбит… прав ием,
 		Пускай вой семья, мне знаю, ж ты,
 		Па


In [259]:

print(evaluate(
    model, 
    char_to_idx, 
    idx_to_char, 
    temp=0.3, 
    prediction_len=500, 
    start_text='V'
    )
)

VI
 
 		В слишь нет на гурали,
 		Не мог Онегин обрести.
 		С ней речь с ней издуслов и хлада и нет.
 
 
 
 IX
 
 		Так поступили без строку
 		Про друзей пустою глупыц
 		Мне смиренный которая,
 		Ее дам, как Ги простой
 		Отец петой устать обо в страшный,
 		На принесется пора самом.
 
 
 
 VIII
 
 		Ее прекрасной старины.
 		Обряд известный угощенья;
 		Своем супругом чепся,
 		Она науком и домашни звучно.
 		Полюбите вы снова: но…
 		Учитесь властвовать собою:
 		Не всякий вас, как уж теки ле


In [261]:

print(evaluate(
    model, 
    char_to_idx, 
    idx_to_char, 
    temp=0.9, 
    prediction_len=500, 
    start_text='V'
    )
)

VII
 
 		В кибитках, чевой постусь холодный
 		Придет ясно ревнивой, сень;
 		Мечтами скрих пора норно?
 		Приминало мне луне;
 		Проказы звезда в Лала;
 		Поеде, вот – и ричил он?
 
 
 
 XXVII
 
 		«Мой секундант? – сказал Евгений,
 		Не привлекла б она очей.
 		Дика, милый порей прочлять
 		Или к ним былым едут свои,
 		И добры чистой серебре,
 		И, понимой легкой отради,
 		С роплушама
 		Потом на бы злун. и влю дребитмая
 		Им разе замела,
 		Как торзие чугат;
 		Чтобы проходили толтна.
 
 
 


Вывод:

Первый подход(генерация фраз) выглядит более предпочтительно, так как создаются +- осмысленные тексты за счет сохранения контекста целых фраз. При генерации отдельных букв могут возникать странные несуществующие слова, зато сохраняется первоначальная структура и стилистика текста.

